# Workbook 1 — In the misty beginnings: intelligence and alignment, then and now

**Solutions · Mysteries of Deep Learning, Iliad Intensive 2026**

Before deep learning, AI had three working pictures of intelligence: search over symbols, sequential decision and control, and statistical learning. This workbook looks at why the third took over, and at what alignment meant before and after capable systems came to be learned rather than written. Most sections are short guides to readings. The one derivation, planning as inference (§3), is a chain of exercises with worked answers; under stated assumptions, it gives planning and maximum-entropy RL a common formulation as inference in one graphical model.

**Route.** The sections are independent; start wherever interests you.
1. [Three pictures of intelligence](#wb1-section-1): a table of the three pictures and what each needs written by hand, and one question. Short.
2. [Why GOFAI failed](#wb1-section-2): one exponential blowup under two names, and Sutton's bitter lesson. Short, one core reading.
3. [Why statistical learning at all? Planning as inference](#wb1-section-3): five exercises, following Levine's tutorial, that derive the soft Bellman equation, show why exact inference makes a reckless agent, and recover maximum-entropy RL and KL-regularized control. It does not settle which route scales (see §3.C). The longest section; pen and paper help.
4. [Why alignment is hard, then](#wb1-section-4): the ontology problem as a short formal sketch, and natural abstractions. Short, one core reading.
5. [Why alignment is hard, now](#wb1-section-5): three readings on what a learned agent's values are, and a pointer to simulators in WB3. Readings only.

Readings marked *core* are the ones most worth your time within their section; everything else is optional. Worked answers to every question are in the solutions notebook, which is there for everyone to use.

<a id="wb1-section-1"></a>

## 1. Three pictures of intelligence

Classical AI had three formal pictures of intelligence, each needing one ingredient written by hand, and much of its history is a sequence of bets on which ingredient could instead be supplied by data and compute.

| picture | formal object | written by hand |
|---|---|---|
| search over symbols | state space $S$, operators, goal test, heuristic $h:S\to\mathbb R_{\ge0}$ (e.g. A* expands states in order of cost-so-far plus $h$) | $S$ and the operators, often $h$ |
| sequential decision and control | Markov decision process $(S,A,P,r,\gamma)$ and its Bellman optimality equation | the reward $r$; a Markov state space |
| statistical learning | hypothesis class $\mathcal H$, loss $\ell$, data distribution $\mathcal D$; empirical risk minimization $\hat h\in\arg\min_{h\in\mathcal H}\frac1n\sum_{i}\ell(h(x_i),y_i)$ (made precise in WB2) | $\mathcal H$: the architecture, and whatever shapes which functions are easy to reach |

**Question 1.1.** Place AlphaZero and a pretrained-then-RL-fine-tuned language model in this taxonomy. For each component (search, sequential decision, learned function), say what plays that role and what is still written by hand. Which hand-written ingredient did each system manage to replace with data?

- **AlphaZero.** *Search:* MCTS, used both to generate self-play training targets and at play time. *Sequential decision:* the game, with a hand-written reward (win/loss/draw) and an exact, known simulator. *Learned function:* the policy/value network, trained to match the MCTS-improved move distribution and the game outcome. *Still hand-written:* the rules (which give a perfect model and a Markov state for free), the reward, the network architecture and the search algorithm. *Replaced by data:* the heuristic. The value network is a learned heuristic, and it is learnable here because the reward and the model cost nothing in a board game.
- **Language model.** *Learned function:* the transformer, trained on next-token prediction over human text, so pretraining needs no reward and no hand-written state space. *Sequential decision:* RL fine-tuning against preference models or verifiers, where the reward is again expensive and hand-supplied. *Search:* sampling, chain-of-thought and best-of-$n$ at test time. *Still hand-written:* the architecture, the tokenizer, data curation, and the RL reward and rubrics. *Replaced by data:* most of the representation, and with it much of what a heuristic or state space would have encoded, supplied by the cheapest data there is (unlabeled text) through one generic $\mathcal H$.
- The contrast: both systems moved the heuristic onto learning, but only the language model moved the representation onto unlabeled data. Neither got rid of the hand-written reward; the LLM kept a smaller but crucial one on top. That asymmetry is where §4–5 pick up.

<a id="wb1-section-2"></a>

## 2. Why GOFAI failed

A search tree with branching factor $b$ has $\sim b^d$ lines of play to depth $d$, and a tabular value function over $k$ state variables at $m$ levels each has $m^k$ entries. Lighthill's "combinatorial explosion" (1973) and Bellman's "curse of dimensionality" are the same blowup, and hand-written structure beats it only one problem at a time.

**[The bitter lesson](http://www.incompleteideas.net/IncIdeas/BitterLesson.html)**, Rich Sutton, 2019 · *core*

Across 70 years of AI research (chess, Go, speech, vision), general methods that leverage computation have beaten methods that build in human domain knowledge, by a large margin, because the cost of computation keeps falling exponentially. The two methods that "seem to scale arbitrarily" are search and learning. Two pages. The last two paragraphs carry the normative claim: build in the meta-methods that find structure, not the structure we have found.

**Question 2.1.** Sutton's vision paragraph credits modern networks that "use only the notions of convolution and certain kinds of invariances", while his final paragraph says we should stop building in simple ways to think about "space, objects, multiple agents, or symmetries". Current systems still contain a lot of hand-written structure: architectures (convolution, attention), tokenizers, curated data mixtures, RL reward rubrics, constitutions. Pick two. Is each a counterexample to the bitter lesson, or just the part that has not yet been eaten?

A useful dividing line is between structure that **scales with compute** and structure that **substitutes for compute**.

- **Convolution or attention.** These encode a symmetry or a routing mechanism, not domain facts, and they make learning cheaper without capping it, so they are arguably "meta-methods" in Sutton's sense. Sutton's own text is pulled both ways here: he praises convolution and invariances in vision, then lists symmetries among the things not to build in. The evidence that architectures converge in performance at scale (WB5 §1) suggests their advantage shrinks as compute grows: built-in structure that does get eaten, just slowly.
- **Tokenizer.** Hand-written, and visibly brittle on arithmetic and spelling. Byte-level models are what the bitter lesson predicts; whether they win at equal compute is still an empirical question.
- **Data curation, RL rubrics, constitutions.** These specify *what we want*, not *how to compute it*. The bitter lesson is a claim about capability, so these may be outside its scope altogether: more compute cannot tell you which outcome is good. This is the thread §4–5 pick up.

<a id="wb1-section-3"></a>

## 3. Why statistical learning at all? Planning as inference

**[Reinforcement learning and control as probabilistic inference: tutorial and review](https://arxiv.org/abs/1805.00909)**, Sergey Levine, 2018 · *core*

Maximum-entropy RL is equivalent to exact probabilistic inference in a graphical model with optimality variables when the dynamics are deterministic, and to variational inference when they are stochastic; maximum-entropy policy gradients, actor-critic methods and soft Q-learning then fall out as approximate inference. The exercises follow §2.2–2.4 and §3.1–3.2; §4 turns the result into soft Q-learning and actor-critic algorithms.

**The result.** Condition a model of trajectories on binary "optimality" variables with likelihood $e^{r}$. The posterior reweights the trajectory prior by $e^{R(\tau)}$; with deterministic dynamics and a fixed start state it is a Boltzmann distribution over feasible plans whose most probable trajectory is the classical optimal plan (Exercise 3.1). Its log-messages obey a *soft* Bellman equation (3.2), which under stochastic dynamics is optimistic (3.3). Holding the dynamics fixed gives variational inference, whose objective is maximum-entropy RL (3.4), and the one-step form is KL-regularized control, which has the form of a Bayesian update (3.5). This is a common formulation of planning and maximum-entropy RL under these assumptions. It does not make fitting a generative model to observations the same objective as control, which still needs a reward or goal.

The exercises build on each other, but each states what it uses. If you only do some, Exercises 3.3 and 3.5 have the most payoff.

### 3.A. The model

Take a finite-horizon MDP with finitely many states and actions (sums in place of Levine's integrals), and trajectories $\tau=(s_1,a_1,\dots,s_T,a_T)$. The "prior" over trajectories is the dynamics together with a uniform action prior $p(a\mid s)=1/|A|$:

$$
p(\tau)=p(s_1)\prod_{t=1}^{T-1}p(s_{t+1}\mid s_t,a_t)\,\prod_{t=1}^{T}\frac{1}{|A|}.
$$

At each step add a binary variable $\mathcal O_t$ ("step $t$ was optimal") with

$$
p(\mathcal O_t=1\mid s_t,a_t)=e^{\,r(s_t,a_t)} .
$$

For this to be a probability we need $r\le0$. Subtracting the maximum reward from every reward arranges that, and since every trajectory has $T$ steps it shifts every total reward by the same amount. (Levine §2.5 shows how to drop the assumption entirely.) Write $\mathcal O$ for the event $\mathcal O_1=\dots=\mathcal O_T=1$, write $\mathcal O_{t:T}$ for $\mathcal O_t=\dots=\mathcal O_T=1$, and let $R(\tau)=\sum_{t=1}^T r(s_t,a_t)$.

For comparison, the ordinary (hard) **Bellman optimality equation** for this finite-horizon, undiscounted problem is

$$
V^*_t(s)=\max_{a\in A}\Big[r(s,a)+\sum_{s'}p(s'\mid s,a)\,V^*_{t+1}(s')\Big],\qquad V^*_{T+1}\equiv0,
$$

the counterpart of the infinite-horizon discounted MDP equation $V^*(s)=\max_a\big[r(s,a)+\gamma\sum_{s'}P(s'\mid s,a)V^*(s')\big]$. Exercises 3.2(d) and 3.5(b) ask when the soft recursions reduce to it.

### 3.B. Five exercises

**Exercise 3.1 — the posterior over trajectories.**

(a) Show that $p(\tau\mid\mathcal O)\propto p(s_1)\prod_{t=1}^{T-1}p(s_{t+1}\mid s_t,a_t)\;e^{R(\tau)}$.

(b) Suppose the dynamics are deterministic, $s_{t+1}=f(s_t,a_t)$, and the start state is fixed. Describe $p(\tau\mid\mathcal O)$. Which trajectory is most probable?

(c) Introduce a temperature $\alpha>0$ by using $r/\alpha$ in place of $r$. What happens to the posterior in (b) as $\alpha\to0$?

<details><summary>Hint</summary>

Bayes' rule: $p(\tau\mid\mathcal O)=p(\mathcal O\mid\tau)\,p(\tau)/p(\mathcal O)$. Given the whole trajectory, each $\mathcal O_t$ depends only on $(s_t,a_t)$. Which factors are the same for every $\tau$?
</details>

**(a)** By Bayes' rule, $p(\tau\mid\mathcal O)=p(\mathcal O\mid\tau)\,p(\tau)/p(\mathcal O)$. Given $\tau$, the $\mathcal O_t$ are independent and each depends only on $(s_t,a_t)$, so

$$
p(\mathcal O\mid\tau)=\prod_{t=1}^T e^{r(s_t,a_t)}=e^{R(\tau)} .
$$

The action-prior factor $|A|^{-T}$ and the evidence $p(\mathcal O)$ are the same for every $\tau$, so

$$
p(\tau\mid\mathcal O)=\frac{|A|^{-T}}{p(\mathcal O)}\,\Big[p(s_1)\prod_{t=1}^{T-1}p(s_{t+1}\mid s_t,a_t)\Big]\,e^{R(\tau)}
\;\propto\;\Big[p(s_1)\prod_{t=1}^{T-1}p(s_{t+1}\mid s_t,a_t)\Big]\,e^{R(\tau)} .
$$

This is Levine's eq. (4): the probability of the trajectory under the dynamics, times its exponentiated total reward. Keep the exact normalized form; Exercise 3.4 uses it.

**(b)** With deterministic dynamics and a fixed start, the bracket is $1$ if $\tau$ is feasible ($s_1$ is the start state and every $s_{t+1}=f(s_t,a_t)$) and $0$ otherwise. So

$$
p(\tau\mid\mathcal O)=\frac{\mathbb 1[\tau\text{ feasible}]\;e^{R(\tau)}}{\sum_{\tau'\,\text{feasible}}e^{R(\tau')}} ,
$$

a Boltzmann distribution over feasible plans with energy $-R$. The most probable trajectory is the feasible plan with the largest total reward, which is exactly the classical optimal plan. **Classical planning is MAP inference in this model.**

**(c)** With $r/\alpha$ the posterior on feasible plans is $\propto e^{R(\tau)/\alpha}$. For two feasible plans with $R(\tau)<R(\tau^*)$,

$$
\frac{p(\tau\mid\mathcal O)}{p(\tau^*\mid\mathcal O)}=e^{-\left(R(\tau^*)-R(\tau)\right)/\alpha}\;\longrightarrow\;0\qquad(\alpha\to0),
$$

so the posterior concentrates, uniformly, on the set of optimal plans. At $\alpha>0$ the full posterior is "soft" planning: near-optimal plans keep weight in proportion to $e^{R/\alpha}$.

**Exercise 3.2 — backward messages and the soft Bellman equation.** Define the backward messages

$$
\beta_t(s,a)=p(\mathcal O_{t:T}\mid s_t=s,a_t=a),\qquad \beta_t(s)=p(\mathcal O_{t:T}\mid s_t=s).
$$

(a) Show that $\beta_T(s,a)=e^{r(s,a)}$ and, for $t<T$,

$$
\beta_t(s,a)=e^{r(s,a)}\sum_{s'}p(s'\mid s,a)\,\beta_{t+1}(s'),\qquad \beta_t(s)=\frac{1}{|A|}\sum_a\beta_t(s,a).
$$

(b) Show that the posterior policy is $p(a_t=a\mid s_t=s,\mathcal O)=\beta_t(s,a)\big/\sum_{a'}\beta_t(s,a')$.

(c) Define $Q_T=r$ and, going backward,

$$
V_t(s)=\log\sum_a e^{Q_t(s,a)},\qquad Q_{t}(s,a)=r(s,a)+\log\sum_{s'}p(s'\mid s,a)\,e^{V_{t+1}(s')}.
$$

Show that $Q_t(s,a)=\log\beta_t(s,a)+c_t$ for a constant $c_t$ that does not depend on $s$ or $a$, and that the posterior policy is $e^{Q_t(s,a)-V_t(s)}$. These are Levine's soft $Q$- and value functions.

(d) Compare with the Bellman optimality equation in §3.A. Put in a temperature by using $r/\alpha$, and rescale $\tilde Q=\alpha Q$, $\tilde V=\alpha V$, so that

$$
\tilde V_t(s)=\alpha\log\sum_a e^{\tilde Q_t(s,a)/\alpha},\qquad \tilde Q_t(s,a)=r(s,a)+\alpha\log\sum_{s'}p(s'\mid s,a)\,e^{\tilde V_{t+1}(s')/\alpha}.
$$

What do the two soft operations become as $\alpha\to0$, first for deterministic dynamics and then for stochastic dynamics?

<details><summary>Hint for (a) and (b)</summary>

For (a), split $\mathcal O_{t:T}$ into $\mathcal O_t$ and $\mathcal O_{t+1:T}$ and sum over $s_{t+1}$. Given $(s_t,a_t)$, $\mathcal O_t$ is independent of everything later, and everything later depends on $(s_t,a_t)$ only through $s_{t+1}$. For (b), apply Bayes' rule to $a_t$ with $s_t$ fixed, then argue that once you know $s_t$, $\mathcal O_{1:t-1}$ is independent of $(a_t,\mathcal O_{t:T})$.
</details>

<details><summary>Hint for (c)</summary>

Induct backward from $t=T$. If $Q_{t+1}=\log\beta_{t+1}+c_{t+1}$, what is $V_{t+1}$ in terms of $\beta_{t+1}(s')$?
</details>

<details><summary>Hint for (d)</summary>

For numbers $x_1,\dots,x_n$ and weights $p_i>0$: $\;\max_i x_i+\alpha\log\min_i p_i\;\le\;\alpha\log\sum_i p_i e^{x_i/\alpha}\;\le\;\max_i x_i+\alpha\log\sum_i p_i$.
</details>

**(a)** At $t=T$ there is one optimality variable, so $\beta_T(s,a)=p(\mathcal O_T=1\mid s,a)=e^{r(s,a)}$. For $t<T$,

$$
\begin{aligned}
\beta_t(s,a)&=p(\mathcal O_t\mid s,a)\;p(\mathcal O_{t+1:T}\mid s,a)
&&\text{($\mathcal O_t$ is independent of the future given $s_t,a_t$)}\\
&=e^{r(s,a)}\sum_{s'}p(s'\mid s,a)\;p(\mathcal O_{t+1:T}\mid s_{t+1}=s')
&&\text{(the future depends on $s_t,a_t$ only through $s_{t+1}$)}\\
&=e^{r(s,a)}\sum_{s'}p(s'\mid s,a)\;\beta_{t+1}(s').
\end{aligned}
$$

Averaging over the action prior gives $\beta_t(s)=\sum_a p(a\mid s)\,\beta_t(s,a)=\frac1{|A|}\sum_a\beta_t(s,a)$. This is sum-product message passing backward in time, as in an HMM (Levine eq. 6).

**(b)** Given $s_t$, everything earlier (including $\mathcal O_{1:t-1}$) is independent of $a_t$ and of everything after it: the dynamics are Markov and the action prior looks only at $s_t$. So $p(a_t\mid s_t,\mathcal O)=p(a_t\mid s_t,\mathcal O_{t:T})$, and Bayes' rule in $a_t$ gives

$$
p(a\mid s,\mathcal O_{t:T})=\frac{p(\mathcal O_{t:T}\mid s,a)\,p(a\mid s)}{p(\mathcal O_{t:T}\mid s)}
=\frac{\beta_t(s,a)/|A|}{\frac1{|A|}\sum_{a'}\beta_t(s,a')}
=\frac{\beta_t(s,a)}{\sum_{a'}\beta_t(s,a')} .
$$

**(c)** Induct backward. At $t=T$, $Q_T=r=\log\beta_T$, so $c_T=0$. Suppose $Q_{t+1}=\log\beta_{t+1}+c_{t+1}$. Then, using (a),

$$
V_{t+1}(s')=\log\sum_a\beta_{t+1}(s',a)+c_{t+1}=\log\beta_{t+1}(s')+\log|A|+c_{t+1},
$$

and substituting into the definition of $Q_t$,

$$
Q_t(s,a)=r(s,a)+\log\sum_{s'}p(s'\mid s,a)\,\beta_{t+1}(s')+\log|A|+c_{t+1}=\log\beta_t(s,a)+c_t,
$$

with $c_t=c_{t+1}+\log|A|$, so $c_t=(T-t)\log|A|$. The constant is the action prior's $1/|A|$, which Levine also drops. It does not depend on $(s,a)$, so it cancels in the policy from (b):

$$
p(a\mid s,\mathcal O)=\frac{e^{Q_t(s,a)-c_t}}{\sum_{a'}e^{Q_t(s,a')-c_t}}=\frac{e^{Q_t(s,a)}}{e^{V_t(s)}}=e^{Q_t(s,a)-V_t(s)} .
$$

**(d)** *Over actions.* By the hint with all $p_i=1$, $\max_a\tilde Q_t\le\tilde V_t\le\max_a\tilde Q_t+\alpha\log|A|$: the log-sum-exp is a soft max, within $\alpha\log|A|$ of the hard one.

*Deterministic dynamics.* The sum over $s'$ has one term, so $\tilde Q_t(s,a)=r(s,a)+\tilde V_{t+1}(f(s,a))$ exactly. For deterministic dynamics the hard recursion of §3.A reads $V^*_t(s)=\max_a Q^*_t(s,a)$ with $Q^*_t(s,a)=r(s,a)+V^*_{t+1}(f(s,a))$ and $V^*_{T+1}\equiv0$. Since $\tilde Q_t$ itself depends on $\alpha$, take the limit by backward induction. At $t=T$, $\tilde Q_T=Q^*_T=r$. If $0\le\tilde V_{t+1}-V^*_{t+1}\le(T-t)\,\alpha\log|A|$, then the same bounds hold for $\tilde Q_t-Q^*_t$, and the soft max adds at most one more $\alpha\log|A|$, so

$$
0\;\le\;\tilde V_t(s)-V^*_t(s)\;\le\;(T-t+1)\,\alpha\log|A|\;\longrightarrow\;0 .
$$

So for deterministic dynamics the limit is the Bellman optimality equation of §3.A.

*Stochastic dynamics.* Let $p_{\min}$ be the smallest positive transition probability over all $(s,a)$, and apply the hint over the reachable next states, those with $p(s'\mid s,a)>0$:

$$
\max_{s':\,p(s'\mid s,a)>0}\tilde V_{t+1}(s')+\alpha\log p_{\min}\;\le\;\alpha\log\sum_{s'}p(s'\mid s,a)\,e^{\tilde V_{t+1}(s')/\alpha}\;\le\;\max_{s':\,p(s'\mid s,a)>0}\tilde V_{t+1}(s').
$$

Define the limiting recursion

$$
V^+_t(s)=\max_a\Big[r(s,a)+\max_{s':\,p(s'\mid s,a)>0}V^+_{t+1}(s')\Big],\qquad V^+_{T+1}\equiv0 .
$$

The same backward induction, now losing at most $\alpha\log(1/p_{\min})$ at each next-state backup and gaining at most $\alpha\log|A|$ at each soft max, gives

$$
-(T-t)\,\alpha\log(1/p_{\min})\;\le\;\tilde V_t(s)-V^+_t(s)\;\le\;(T-t+1)\,\alpha\log|A|\;\longrightarrow\;0 .
$$

So the backup tends to the best *reachable* next value, **not** the expected one. The zero-temperature limit is therefore not the Bellman equation of §3.A: the agent plans as if it also chose the outcome of every die roll. Even at $\alpha=1$, Jensen's inequality gives $\log\mathbb E\,e^{V}\ge\mathbb E\,V$, so the exact-inference backup is never less optimistic than the expected one. Levine §2.3 (around eq. 7) describes this as risk-seeking; Exercise 3.3 makes it concrete.

**Exercise 3.3 — inference is optimistic.** At some state the agent can take a *safe* action, which leads for certain to a state with soft value $0$, or a *gamble*, which leads with probability $p$ to a state with soft value $+R$ and otherwise to a state with soft value $-C$. Both actions have the same immediate reward, so ignore it. (Adding a constant to every value shifts both backups in (a) by that constant, so taking the safe value to be $0$ loses nothing.)

(a) Compute $Q(\text{gamble})$ under the exact-inference backup of Exercise 3.2, $\log\mathbb E\,e^{V}$, and under the ordinary expected backup, $\mathbb E\,V$.

(b) Evaluate both for $p=0.01$, $R=10$, $C=100$. Which action does each prefer, and by what odds does the exact posterior policy $e^{Q-V}$ pick the gamble over the safe action?

(c) Compute the posterior probability that the gamble came out well, $p(s_{t+1}=\text{good}\mid s_t,a_t=\text{gamble},\mathcal O)$. What has the agent "inferred"?

(d) For large $C$, when does exact inference prefer the gamble? How does the answer depend on $C$?

<details><summary>Hint for (c)</summary>

By Bayes' rule and the Markov property, $p(s_{t+1}\mid s_t,a_t,\mathcal O)\propto p(s_{t+1}\mid s_t,a_t)\,\beta_{t+1}(s_{t+1})$, and by Exercise 3.2(c), $\beta_{t+1}(s')\propto e^{V_{t+1}(s')}$.
</details>

**(a)** The safe action has $Q=0$ under both backups. For the gamble,

$$
Q_{\text{exact}}=\log\big(p\,e^{R}+(1-p)\,e^{-C}\big),\qquad Q_{\text{expected}}=pR-(1-p)C .
$$

**(b)** With $p=0.01$, $R=10$, $C=100$: $p\,e^{R}=0.01\,e^{10}\approx220.3$ and $(1-p)e^{-C}=0.99\,e^{-100}\approx3.7\times10^{-44}$, which is negligible. So

$$
Q_{\text{exact}}\approx\log220.3=10+\log0.01\approx5.39,\qquad Q_{\text{expected}}=0.1-99=-98.9 .
$$

Exact inference prefers the gamble: the posterior policy is a softmax of $Q$, so the odds are $e^{5.39}/e^{0}\approx220$ to $1$ in favour of the gamble. The expected backup rejects it; even a max-ent policy built on it gives odds of about $e^{-98.9}\approx10^{-43}$.

**(c)** Using the hint,

$$
p(\text{good}\mid s_t,\text{gamble},\mathcal O)=\frac{p\,e^{R}}{p\,e^{R}+(1-p)\,e^{-C}}=\frac{1}{1+99\,e^{-110}}\approx1-1.7\times10^{-46}.
$$

Conditioned on "everything went well", the agent has inferred that the gamble paid off. Exact inference tilts the dynamics as well as the actions, so the agent plans as if it could choose its luck. Levine §2.3–2.4 and the opening of §3 make the same point: under the exact posterior, $p(s_{t+1}\mid s_t,a_t,\mathcal O)\neq p(s_{t+1}\mid s_t,a_t)$.

**(d)** Exact inference prefers the gamble when $p\,e^{R}+(1-p)e^{-C}>1$. As $C\to\infty$ this becomes $p\,e^{R}>1$, i.e. $R>\log(1/p)$, which **does not depend on $C$ at all**. However large the catastrophe, any gamble whose win satisfies $R>\log(1/p)$ looks good, however improbable the win. The same caution applies to any scheme that conditions a model of the world on success. Exercise 3.4 removes the problem by forbidding the agent to choose its luck.

**Exercise 3.4 — the variational fix is maximum-entropy RL.** Restrict to trajectory distributions that use the *true* start-state distribution and dynamics, and let only the policy vary. The horizon is finite, so the policy may depend on $t$:

$$
q(\tau)=p(s_1)\prod_{t=1}^{T-1}p(s_{t+1}\mid s_t,a_t)\,\prod_{t=1}^{T}\pi_t(a_t\mid s_t).
$$

(a) Show that

$$
\mathrm{KL}\big(q\,\|\,p(\cdot\mid\mathcal O)\big)=-\,\mathbb E_q\Big[\sum_{t=1}^T\Big(r(s_t,a_t)+\mathcal H\big(\pi_t(\cdot\mid s_t)\big)\Big)\Big]+T\log|A|+\log p(\mathcal O),
$$

where $\mathcal H(\pi_t(\cdot\mid s))=-\sum_a\pi_t(a\mid s)\log\pi_t(a\mid s)$ is the policy's entropy at $s$.

(b) Conclude that minimizing this KL over the policies is the same as maximizing expected reward plus policy entropy (maximum-entropy RL), and that the expectation minus $T\log|A|$ is a lower bound on $\log p(\mathcal O)$.

(c) When is the minimum KL zero, so that max-ent RL is *exact* inference? Use Exercise 3.3(c) to say why it is not, in general.

<details><summary>Hint</summary>

Write out $\log q(\tau)-\log p(\tau\mid\mathcal O)$ using the exact normalized posterior from Exercise 3.1(a),

$$
p(\tau\mid\mathcal O)=\frac{|A|^{-T}}{p(\mathcal O)}\,p(s_1)\prod_{t=1}^{T-1}p(s_{t+1}\mid s_t,a_t)\,e^{R(\tau)} .
$$

The start-state and dynamics factors cancel. For the policy term, $\mathbb E_q[\log\pi_t(a_t\mid s_t)]$ averages $\sum_a\pi_t(a\mid s_t)\log\pi_t(a\mid s_t)$ over the distribution of $s_t$.
</details>

**(a)** From Exercise 3.1(a), $p(\tau\mid\mathcal O)=|A|^{-T}\,p(s_1)\prod_t p(s_{t+1}\mid s_t,a_t)\,e^{R(\tau)}/p(\mathcal O)$. Since $q$ uses the same $p(s_1)$ and the same dynamics, those factors cancel in the log-ratio:

$$
\log q(\tau)-\log p(\tau\mid\mathcal O)=\sum_{t=1}^T\log\pi_t(a_t\mid s_t)+T\log|A|-R(\tau)+\log p(\mathcal O).
$$

Take the expectation under $q$. For each $t$,

$$
\mathbb E_q[\log\pi_t(a_t\mid s_t)]=\mathbb E_{s_t\sim q}\Big[\sum_a\pi_t(a\mid s_t)\log\pi_t(a\mid s_t)\Big]=-\,\mathbb E_q\big[\mathcal H(\pi_t(\cdot\mid s_t))\big],
$$

which gives the stated formula. Levine's eq. (11) is the same statement with the constants dropped.

**(b)** The last two terms do not depend on the policies, so minimizing the KL is maximizing $\mathbb E_q\big[\sum_t\big(r+\mathcal H(\pi_t(\cdot\mid s_t))\big)\big]$, the maximum-entropy RL objective. Since $\mathrm{KL}\ge0$,

$$
\log p(\mathcal O)\;\ge\;\mathbb E_q\Big[\sum_t\big(r(s_t,a_t)+\mathcal H(\pi_t(\cdot\mid s_t))\big)\Big]-T\log|A|,
$$

an evidence lower bound (Levine eq. 19, up to the constant). Note also that $\log|A|-\mathcal H(\pi)=\mathrm{KL}(\pi\,\|\,\mathrm{Unif}_A)$, so the entropy bonus is really a KL penalty toward the action prior. Exercise 3.5 replaces the uniform prior by a general one.

**(c)** The minimum is zero exactly when the true posterior lies in the family. The true posterior factorizes as $p(s_1\mid\mathcal O)\prod_t p(s_{t+1}\mid s_t,a_t,\mathcal O)\prod_t p(a_t\mid s_t,\mathcal O)$ (Levine eq. 9). By Exercise 3.3(c), $p(s_{t+1}\mid s_t,a_t,\mathcal O)\propto p(s_{t+1}\mid s_t,a_t)\,\beta_{t+1}(s_{t+1})$, which differs from the true dynamics whenever $\beta_{t+1}$ is not constant on the reachable next states.

- With **deterministic dynamics and a fixed start state** there is only one possible next state, so the tilt does nothing. The posterior is in the family, with $\pi_t$ equal to the posterior policy $e^{Q_t-V_t}$ of Exercise 3.2 (which is why the policy needs its time index), and max-ent RL is exact inference.
- With **stochastic dynamics** the posterior generally lies outside the family, and max-ent RL is variational inference with the dynamics held fixed: the agent may choose its actions but not its luck.

This is the statement in Levine's abstract.

**Exercise 3.5 — KL-regularized control as a Bayesian update.** Let $A$ be finite, let $\pi_0$ be a reference distribution on $A$ with full support, and let $r:A\to\mathbb R$.

(a) Let $Z=\sum_a\pi_0(a)e^{r(a)}$ and $\pi^*(a)=\pi_0(a)e^{r(a)}/Z$. Show that for every distribution $\pi$ on $A$,

$$
\mathbb E_\pi[r]-\mathrm{KL}(\pi\,\|\,\pi_0)=\log Z-\mathrm{KL}(\pi\,\|\,\pi^*).
$$

Conclude that $\max_\pi\{\mathbb E_\pi[r]-\mathrm{KL}(\pi\|\pi_0)\}=\log\mathbb E_{\pi_0}e^{r}$, attained only at $\pi^*\propto\pi_0e^{r}$. What changes for $\mathbb E_\pi[r]-\lambda\,\mathrm{KL}(\pi\|\pi_0)$ with a coefficient $\lambda>0$? (RLHF papers, and WB3, usually write this coefficient as $\beta$; here $\beta$ is taken by the backward messages.)

(b) Use (a) to solve the max-ent objective of Exercise 3.4 backward in time. Let $V_t(s)$ be the best achievable expected future reward-plus-entropy from state $s$ at time $t$, with $V_{T+1}\equiv0$. Show that the optimal policy is $\pi_t(a\mid s)=e^{Q_t(s,a)-V_t(s)}$, where

$$
Q_t(s,a)=r(s,a)+\mathbb E_{s'\sim p(\cdot\mid s,a)}V_{t+1}(s'),\qquad V_t(s)=\log\sum_a e^{Q_t(s,a)}.
$$

Compare with Exercise 3.2(c), redo Exercise 3.3(b) with this backup, and take the zero-temperature limit as in Exercise 3.2(d).

(c) Read $\pi^*\propto\pi_0e^{r}$ as Bayes' rule. What is the prior, what is the likelihood, and what is $Z$?

<details><summary>Hint for (a)</summary>

Expand both KLs as sums over $a$ and substitute $\log\pi^*(a)=\log\pi_0(a)+r(a)-\log Z$.
</details>

<details><summary>Hint for (b)</summary>

$\mathcal H(\pi)=\log|A|-\mathrm{KL}(\pi\,\|\,\mathrm{Unif}_A)$. Work from $t=T$ backward. Once the later policies are optimal, the part of the objective that depends on $\pi(\cdot\mid s_t)$ is $\mathbb E_{a\sim\pi}\big[r(s_t,a)+\mathbb E_{s'}V_{t+1}(s')\big]+\mathcal H(\pi)$.
</details>

**(a)** Expand the right-hand KL and substitute $\log\pi^*=\log\pi_0+r-\log Z$:

$$
\mathrm{KL}(\pi\,\|\,\pi^*)=\sum_a\pi(a)\log\pi(a)-\sum_a\pi(a)\big[\log\pi_0(a)+r(a)-\log Z\big]=\mathrm{KL}(\pi\,\|\,\pi_0)-\mathbb E_\pi[r]+\log Z .
$$

Rearranging gives the identity. Since $\mathrm{KL}(\pi\|\pi^*)\ge0$ with equality only at $\pi=\pi^*$, the maximum is $\log Z=\log\mathbb E_{\pi_0}e^{r}$, attained only at $\pi^*$. With a coefficient $\lambda$, divide the objective by $\lambda$ to get $\mathbb E_\pi[r/\lambda]-\mathrm{KL}(\pi\|\pi_0)$. The maximizer is $\pi^*\propto\pi_0e^{r/\lambda}$, and the optimal value is $\lambda\log\mathbb E_{\pi_0}e^{r/\lambda}$. As $\lambda\to0$, $\pi^*$ concentrates on the reward-maximizing actions (all of $A$ is in the support of $\pi_0$); as $\lambda\to\infty$ it stays at $\pi_0$.

**(b)** *Last step.* From state $s$ at $t=T$ the objective is $\mathbb E_{a\sim\pi}[r(s,a)]+\mathcal H(\pi)=\mathbb E_\pi[r(s,\cdot)]-\mathrm{KL}(\pi\|\mathrm{Unif}_A)+\log|A|$. By (a) with $\pi_0=\mathrm{Unif}_A$, the maximizer is $\pi_T(a\mid s)\propto e^{r(s,a)}$ and the value is $\log\frac1{|A|}\sum_a e^{r(s,a)}+\log|A|=\log\sum_a e^{r(s,a)}$. That is the claim at $t=T$, with $Q_T=r$.

*Inductive step.* Suppose the policies from $t+1$ on are optimal, with value $V_{t+1}$. Choosing $\pi(\cdot\mid s)$ at time $t$ affects only which $(a,s_{t+1})$ we land in, and from $s_{t+1}$ the best achievable continuation is $V_{t+1}(s_{t+1})$ regardless of how we got there. So from $s_t=s$ the objective is

$$
\mathbb E_{a\sim\pi}\Big[\underbrace{r(s,a)+\mathbb E_{s'}V_{t+1}(s')}_{Q_t(s,a)}\Big]+\mathcal H(\pi)=\mathbb E_\pi[Q_t(s,\cdot)]-\mathrm{KL}(\pi\|\mathrm{Unif}_A)+\log|A|,
$$

and (a) again gives $\pi_t\propto e^{Q_t(s,\cdot)}$ with value $\log\sum_a e^{Q_t(s,a)}=V_t(s)$. These are Levine's eqs. (15)–(16).

*Comparison.* The soft max over actions is the same as in Exercise 3.2(c), but the backup over next states is now an ordinary expectation, $\mathbb E\,V$, rather than $\log\mathbb E\,e^{V}$. In Exercise 3.3 the gamble now gets $Q=pR-(1-p)C=-98.9$, and the optimism is gone. With a temperature (rewards $r/\alpha$, values rescaled by $\alpha$), the action soft max tends to a max (by the same backward induction as in Exercise 3.2(d)) while the expectation stays an expectation. So the zero-temperature limit is the Bellman optimality equation of §3.A, now for stochastic dynamics too.

**(c)** $\pi_0$ is the prior, and the factor $e^{r}$ (or $e^{r/\lambda}$) plays the role of a likelihood for the "observation" that the action was optimal, just like $p(\mathcal O=1\mid a)=e^{r(a)}$ in §3.A; $\pi^*$ is the posterior. To make the likelihood a genuine probability, choose $b\ge\max_a r(a)$ (possible since $A$ is finite) and set $p(\mathcal O=1\mid a)=e^{r(a)-b}$. This leaves $\pi^*$ unchanged, and the evidence is $p(\mathcal O=1)=e^{-b}Z$, so the optimal value $\log Z$ is the log-evidence up to the constant $b$.

So maximizing reward while staying KL-close to a reference policy has the form of a Bayesian update of that policy. Note the scope: this closed form is a *one-step* result. It applies to RL fine-tuning of a pretrained model with a KL penalty when each whole completion (for a fixed prompt) is treated as one action, which is how WB3 §6 uses it. For sequential, token-by-token control with a reference policy $\pi_0(a\mid s)$ at each step, the objective $\mathbb E_q\sum_t\big[r(s_t,a_t)-\mathrm{KL}(\pi_t(\cdot\mid s_t)\,\|\,\pi_0(\cdot\mid s_t))\big]$ is solved by the backward induction of (b) with $\pi_0$ in place of $\mathrm{Unif}_A$: $Q_t$ is as in (b), $V_t(s)=\log\sum_a\pi_0(a\mid s)\,e^{Q_t(s,a)}$, and $\pi_t=\pi_0\,e^{Q_t-V_t}$. The immediate reward is replaced by $Q_t$, which includes the value of what follows.

### 3.C. Where this leaves statistical learning

In this model, planning does inference afresh at decision time from a known $p(s'\mid s,a)$, model-free RL amortizes the posterior into a policy trained from scalar rewards, and generative modelling fits a model of trajectories, including a prior over behaviour like the $\pi_0$ of Exercise 3.5, with no reward, leaving the conditioning on what we want (prompting, goal-conditioning, or the reweighting of Exercise 3.5) for later. The derivation does not say which of these makes better use of data and compute: WB3 §1 compares code lengths for labels and raw text, and WB2 §2 takes up the no-free-lunch objection to the learning route.

<a id="wb1-section-4"></a>

## 4. Why alignment is hard, then

Classical alignment meant writing a utility over the agent's world model; the ontology problem is that the model can change, leaving the old utility with several extensions.

**[Testing the natural abstraction hypothesis: project intro](https://www.lesswrong.com/posts/cy3BhHrGinZCp3LXE/testing-the-natural-abstraction-hypothesis-project-intro)**, John Wentworth, 2021 · *core*

Proposes the natural abstraction hypothesis: information about a system that matters far away has a low-dimensional summary (*abstractability*); humans use roughly those summaries (*human-compatibility*); and diverse learners converge on them (*convergence*). Argues alignment becomes much easier if it holds, and plans simulation tests. Short. WB5 asks related questions about representational convergence across networks.

**Setup for Question 4.1.** Let $U:S\to\mathbb R$ be a utility on the states of the agent's world model, and let a refined world model have states $S'$ with a map $\phi:S'\to S$ that is ambiguous on a region $E\subseteq S'$: each $s'\in E$ has two plausible images $\phi_1(s')\neq\phi_2(s')$. (For example, $U$ counts diamonds as lattices of carbon atoms, the refined model is about quantum fields, and $E$ holds edge cases such as exotic isotopes or simulated diamonds that the old ontology never had to classify.) Then $U_1=U\circ\phi_1$ and $U_2=U\circ\phi_2$ are both reasonable translations of $U$ to $S'$. They agree off $E$, may disagree on $E$, and nothing in $U$ chooses between them. An optimizer can still be drawn into $E$: if some states in $E$ are mapped by some $\phi_i$ to high-utility states but are cheaper to reach than ordinary ones (a simulated diamond is easier to make than a real one), then maximizing that $U_i$ over achievable states selects them, which is exactly where the choice of translation matters.

**Question 4.1.** In the diamond example, where does the ambiguity of $\phi$ come from? Which of Wentworth's three claims would have to hold for the ambiguity to be harmless, and is there residual ambiguity that even a true natural abstraction hypothesis would leave?

- **Where it comes from.** The old concept was defined by examples and rough boundaries that worked in the old ontology. The new ontology contains states the old concept never had to classify. It is a generalization problem: the concept was, in effect, fit on an old distribution and is now queried off it.
- **What has to hold.** *Convergence* and *human-compatibility* do the work. If the agent's refined model contains a variable that matches the human "diamond", then $\phi$ is canonical on ordinary states. *Abstractability* alone only says that *some* low-dimensional summary exists, not that it is ours.
- **What is left over.** Natural abstractions are approximate and defined for *typical* states. An optimizer selects whichever achievable states score highest, and the cheapest high scorers are often atypical edge cases where even natural abstractions blur. So the hypothesis shrinks $E$ but does not stop optimization from selecting states in it.

<a id="wb1-section-5"></a>

## 5. Why alignment is hard, now

Three readings on what a learned agent's values are; WB3 §6 adds a fourth, simulators and personae (janus; Nardo).

### 5.A. Values in a brain-like learner

**[Intro to brain-like AGI safety (sequence; posts 1, 3, 6 and 9)](https://www.lesswrong.com/s/HzcM2dkCq7fwXBej8)**, Steven Byrnes, 2022 (revised since) · *optional*

If AGI is built like the brain, as model-based RL with a learned world model plus a small hard-wired steering subsystem that supplies reward, then its values are a learned value function over concepts in its learned world model. They are neither its reward function nor reward-maximization, and alignment becomes a question of how reward and credit assignment shape them. For values, read [post 1](https://www.lesswrong.com/posts/4basF9w9jaPZpoC8R/intro-to-brain-like-agi-safety-1-what-s-the-problem-and-why) (the problem), [post 3](https://www.lesswrong.com/posts/hE56gYi5d68uux9oM/intro-to-brain-like-agi-safety-3-two-subsystems-learning-and) (learning and steering subsystems), [post 6](https://www.lesswrong.com/posts/qNZSBqLEh4qLRqgWW/intro-to-brain-like-agi-safety-6-big-picture-of-motivation) (motivation and decision-making) and [post 9](https://www.lesswrong.com/posts/vpdJz4k5BgGzuGo7A/intro-to-brain-like-agi-safety-9-takeaways-from-neuro-2-2-on) (AGI motivation; its §9.2, goals defined over latent variables, is §4's ontology problem seen from inside a learner).

### 5.B. Learned optimization

**[Risks from learned optimization in advanced machine learning systems](https://arxiv.org/abs/1906.01820)**, Evan Hubinger, Chris van Merwijk, Vladimir Mikulik, Joar Skalse, Scott Garrabrant, 2019 · *core*

A trained model may itself be an optimizer (a mesa-optimizer) whose objective differs from the training loss. This splits alignment into outer alignment (loss vs intent) and inner alignment (mesa-objective vs loss); a pseudo-aligned model agrees with the loss on the training distribution and comes apart off it. §1 defines the terms; §2 asks which tasks and base optimizers favour mesa-optimization (§2.1 argues that diverse environments favour learned search: §2's bitter lesson applied *inside* the model); §4 on deceptive alignment is optional; §7 is a glossary.

### 5.C. "Just retarget the search"

**[How to go from interpretability to alignment: just retarget the search](https://www.lesswrong.com/posts/w4aeAFzSAguvqA5qu/how-to-go-from-interpretability-to-alignment-just-retarget-the-search)**, John Wentworth, 2022 · *optional*

Assume (i) some version of the natural abstraction hypothesis, so the model has an internal concept for an alignment target such as human values or user intent, and (ii) the mesa-optimization argument, so the model contains retargetable general-purpose search. Then interpretability could find the search's target and point it at that concept, which removes the inner alignment problem but leaves outer alignment unsolved. Short; read all of it, including the 'Problems' section.

### What carries forward

- **The statistical-learning picture** of §1 is made precise in WB2; WB2 §2 takes up the no-free-lunch objection mentioned in §3.C.
- **The reweighting $\pi^*\propto\pi_0e^{r/\lambda}$** of Exercise 3.5 (with $\lambda$ written $\beta$) is applied to RLHF in WB3 §6, which also develops the simulator picture pointed to in §5.
- **Natural abstractions** (§4) connect to the questions about representational convergence in WB5.
- **The hand-written part of $\mathcal H$** in §1, its inductive bias, is taken apart in WB4.

### Further study

- Levine §5 reviews earlier work connecting inference and control (the Kalman duality, linearly solvable MDPs, KL-constrained policy search) and applications such as intent inference.
- The rest of Byrnes's sequence (posts 10–15) covers the technical alignment problem and two paths forward ("controlled AGI" and "social-instinct AGI").